In [1]:
import pandas as pd
from models import crime_serializer, crime_from_row
from kafka import KafkaProducer

In [ ]:
path = "../data/crimes_2025.csv"
columns = ['Date', 'Primary Type', 'Community Area', 'Latitude', 'Longitude']
df = pd.read_csv(path, usecols=columns).head(1000)
df = df.rename(columns={
    'Date': 'event_time',
    'Primary Type': 'crime_type',
    'Community Area': 'community_area',
    'Latitude': 'latitude',
    'Longitude': 'longitude'
})
df.head()

,event_time,crime_type,community_area,latitude,longitude
0,12/31/2025 11:58:00 PM,ASSAULT,61.0,41.802549,-87.667246
1,12/31/2025 11:55:00 PM,MOTOR VEHICLE THEFT,25.0,41.882329,-87.758411
2,12/31/2025 11:54:00 PM,PUBLIC PEACE VIOLATION,76.0,41.976290,-87.905227
3,12/31/2025 11:54:00 PM,BATTERY,28.0,41.885427,-87.661759
4,12/31/2025 11:54:00 PM,PUBLIC PEACE VIOLATION,76.0,41.976290,-87.905227


In [3]:
# Example usage: Convert the first row to a Crime instance
crime = crime_from_row(df.iloc[0])
print(crime)

Crime(event_time='12/31/2025 11:58:00 PM', crime_type='ASSAULT', community_area=61, latitude=41.802549018, longitude=-87.667246428)


In [4]:
server = 'localhost:9092'
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=crime_serializer
)

topic_name = 'crimes'
producer.send(topic_name, value=crime) # Directly converting the dataclass to JSON bytes
producer.flush()

In [5]:
# Send all crimes to Kafka topic in a loop
import time

t0 = time.time()

for _, row in df.iterrows():
    crime = crime_from_row(row)
    producer.send(topic_name, value=crime)
    print(f"Sent: {crime}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

Sent: Crime(event_time='12/31/2025 11:58:00 PM', crime_type='ASSAULT', community_area=61, latitude=41.802549018, longitude=-87.667246428)
Sent: Crime(event_time='12/31/2025 11:55:00 PM', crime_type='MOTOR VEHICLE THEFT', community_area=25, latitude=41.882328854, longitude=-87.758411303)
Sent: Crime(event_time='12/31/2025 11:54:00 PM', crime_type='PUBLIC PEACE VIOLATION', community_area=76, latitude=41.976290414, longitude=-87.905227221)
Sent: Crime(event_time='12/31/2025 11:54:00 PM', crime_type='BATTERY', community_area=28, latitude=41.885426714, longitude=-87.661759042)
Sent: Crime(event_time='12/31/2025 11:54:00 PM', crime_type='PUBLIC PEACE VIOLATION', community_area=76, latitude=41.976290414, longitude=-87.905227221)
Sent: Crime(event_time='12/31/2025 11:50:00 PM', crime_type='BATTERY', community_area=69, latitude=41.77029895, longitude=-87.628293582)
Sent: Crime(event_time='12/31/2025 11:50:00 PM', crime_type='BATTERY', community_area=32, latitude=41.875272573, longitude=-87.6242